In [167]:
from pathlib import Path
import pandas as pd

processed_root = Path(
    r"C:\Users\Ander\Desktop\F1-project\data\processed"
)

parquet_files = list(processed_root.rglob("*.parquet"))

print(f"Found {len(parquet_files)} parquet files")

Found 2159 parquet files


In [ ]:
from pathlib import Path
import pandas as pd

processed_root = Path(
    r"C:\Users\Ander\Desktop\F1-project\data\processed"
)

records = []

EXCLUDE_FILES = {
    "all_race_laps_metadata_enriched.parquet",
    "all_race_laps_metadata_with_times.parquet",
    "all_race_laps_metadata.parquet",
    "all_race_laps_metadata_enriched.parquet"
}


for file in processed_root.rglob("*.parquet"):

    if file.name in EXCLUDE_FILES:
        continue

    parts = file.parts

    filename = file.stem

    # -----------------------------
    # Parse metadata
    # -----------------------------

    try:
        race = parts[-3]
        session = parts[-2]
    except:
        race = None
        session = None

    # -----------------------------
    # Determine file type
    # -----------------------------

    if filename.endswith("_laps"):

        file_type = "laps"
        driver = filename.replace("_laps", "")

    elif filename in [
        "weather",
        "corners",
        "drivers",
        "rcm",
        "session_laptimes",
    ]:

        file_type = filename
        driver = None

    else:

        file_type = "telemetry"
        driver = filename

    records.append({
        "path": str(file),
        "race": race,
        "session": session,
        "type": file_type,
        "driver": driver,
    })

registry = pd.DataFrame(records)

display(registry.head())

print(registry.shape)

,path,race,session,type,driver
0,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,telemetry,ALB
1,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,laps,ALB
2,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,telemetry,ALO
3,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,laps,ALO
4,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,telemetry,ANT


(2156, 5)


In [169]:
# Inspect all telemetry files and load one telemetry dataframe

telemetry_registry = registry[registry["type"] == "telemetry"]


# Load first telemetry parquet
telemetry_path = telemetry_registry.iloc[0]["path"]

telemetry_df = pd.read_parquet(telemetry_path)


print(telemetry_df.columns)

Index(['time', 'rpm', 'speed', 'gear', 'throttle', 'brake', 'drs', 'distance',
       'rel_distance', 'DriverAhead', 'DistanceToDriverAhead', 'acc_x',
       'acc_y', 'acc_z', 'x', 'y', 'z', 'dataKey', 'race', 'session', 'driver',
       'lap'],
      dtype='str')


In [170]:
registry["type"].value_counts()

type
telemetry           958
laps                958
corners              48
drivers              48
rcm                  48
session_laptimes     48
weather              48
Name: count, dtype: int64

In [171]:
telemetry_features = [
    "speed",
    "throttle",
    "brake",
    "gear",
    "rpm",
    "drs",
    "acc_x",
    "acc_y",
    "acc_z",
    "rel_distance"
]

In [172]:
import sys
from pathlib import Path

# go up from notebooks/ to project root
sys.path.append(str(Path.cwd().parent))

In [173]:
from src.data.load_driver_telemetry import load_driver_session

import numpy as np
import pandas as pd

TARGET_POINTS = 500

all_laps = []

for row in registry.itertuples():

    # only race telemetry
    if row.type != "telemetry":
        continue

    if row.session != "Race":
        continue

    print(f"Processing {row.race} | {row.driver}")

    # load telemetry
    telemetry, _ = load_driver_session(
        registry,
        row.race,
        row.session,
        row.driver
    )

    telemetry = telemetry.sort_values(["lap", "time"])

    # process every lap
    for lap_number, lap_df in telemetry.groupby("lap"):

        # remove broken laps
        if len(lap_df) < 500:
            continue

        lap_df = (
            lap_df
            .sort_values("time")
            .drop_duplicates(subset="distance")
            .copy()
        )

        # skip invalid laps
        if lap_df["distance"].nunique() < 2:
            continue

        # normalize distance -> [0, 1]
        x_old = (
            lap_df["distance"] - lap_df["distance"].min()
        ) / (
            lap_df["distance"].max() - lap_df["distance"].min()
        )

        # target interpolation grid
        x_new = np.linspace(0, 1, TARGET_POINTS)

        interp_features = []

        # interpolate telemetry channels
        for col in telemetry_features:

            y_interp = np.interp(
                x_new,
                x_old,
                lap_df[col]
            )

            interp_features.append(y_interp)

        # shape -> (500, num_features)
        lap_tensor = np.stack(
            interp_features,
            axis=1
        )

        all_laps.append({
            "race": row.race,
            "driver": row.driver,
            "lap": lap_number,
            "tensor": lap_tensor
        })

# final ML tensor
X = np.stack([
    item["tensor"] for item in all_laps
])

print("Final tensor shape:")
print(X.shape)

print("Example lap tensor:")
print(X[0])

Processing Abu Dhabi Grand Prix | ALB
Processing Abu Dhabi Grand Prix | ALO
Processing Abu Dhabi Grand Prix | ANT
Processing Abu Dhabi Grand Prix | BEA
Processing Abu Dhabi Grand Prix | BOR
Processing Abu Dhabi Grand Prix | COL
Processing Abu Dhabi Grand Prix | GAS
Processing Abu Dhabi Grand Prix | HAD
Processing Abu Dhabi Grand Prix | HAM
Processing Abu Dhabi Grand Prix | HUL
Processing Abu Dhabi Grand Prix | LAW
Processing Abu Dhabi Grand Prix | LEC
Processing Abu Dhabi Grand Prix | NOR
Processing Abu Dhabi Grand Prix | OCO
Processing Abu Dhabi Grand Prix | PIA
Processing Abu Dhabi Grand Prix | RUS
Processing Abu Dhabi Grand Prix | SAI
Processing Abu Dhabi Grand Prix | STR
Processing Abu Dhabi Grand Prix | TSU
Processing Abu Dhabi Grand Prix | VER
Processing Australian Grand Prix | ALB
Processing Australian Grand Prix | ALO
Processing Australian Grand Prix | ANT
Processing Australian Grand Prix | BEA
Processing Australian Grand Prix | BOR
Processing Australian Grand Prix | DOO
Proces

In [174]:
all_laps_df = pd.DataFrame([
    {
        "idx": i,
        "race": item["race"],
        "driver": item["driver"],
        "lap": item["lap"]
    }
    for i, item in enumerate(all_laps)
])

SAVE_DIR = r"C:/Users/Ander/Desktop/F1-project/data/processed"

# save tensor
np.save(f"{SAVE_DIR}/all_race_laps_tensor.npy", X)

# save metadata
all_laps_df.to_parquet(
    f"{SAVE_DIR}/all_race_laps_metadata.parquet",
    index=False
)

print("Saved successfully")

Saved successfully


In [175]:
from src.data.load_driver_telemetry import load_driver_session

_ , ham = load_driver_session(
    registry,
    "Monaco Grand Prix",
    "Race",
    "HAM"
)

display(ham[ham["lap"] == 1][["time"]])

# load saved files
X = np.load(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_tensor.npy"
)

laps = pd.read_parquet(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_metadata.parquet"
)

# find Monaco HAM lap 1
result = laps[
    (laps["race"] == "Monaco Grand Prix") &
    (laps["driver"] == "HAM") &
    (laps["lap"] == 1)
]

display(result)

idx = result.iloc[0]["idx"]

print(idx)

lap_tensor = X[idx]

print(lap_tensor.shape)

,time
0,89.406


,idx,race,driver,lap
19030,19030,Monaco Grand Prix,HAM,1


19030
(500, 10)


In [176]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# existing data
X = np.load(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_tensor.npy"
)

laps = pd.read_parquet(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_metadata.parquet"
)

lap_times = []

for _, row in tqdm(laps.iterrows(), total=len(laps)):

    try:
        _, session = load_driver_session(
            registry,
            row["race"],
            "Race",
            row["driver"]
        )

        lap_time = session[
            session["lap"] == row["lap"]
        ]["time"].iloc[0]

        lap_times.append(float(lap_time))

    except Exception as e:

        print(
            f"Failed: {row['race']} "
            f"{row['driver']} lap {row['lap']}"
        )

        lap_times.append(np.nan)

laps["lap_time"] = lap_times

  5%|▌         | 1410/26666 [00:48<13:50, 30.42it/s]

Failed: Australian Grand Prix DOO lap 1


  7%|▋         | 1911/26666 [01:06<10:01, 41.17it/s]

Failed: Australian Grand Prix SAI lap 1


  8%|▊         | 2172/26666 [01:14<09:50, 41.51it/s]

...

 80%|███████▉  | 21323/26666 [13:17<02:22, 37.37it/s]

Failed: Saudi Arabian Grand Prix GAS lap 1


 92%|█████████▏| 24638/26666 [14:59<01:03, 31.96it/s]

Failed: São Paulo Grand Prix BOR lap 1


100%|██████████| 26666/26666 [15:57<00:00, 27.84it/s]


In [177]:
valid_mask = ~laps["lap_time"].isna()

laps = laps[valid_mask].reset_index(drop=True)

X = X[valid_mask.values]

In [178]:
laps.to_parquet(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_metadata_with_times.parquet"
)

np.save(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_tensor_clean.npy",
    X
)